# ROI-Based Image Feature Extraction for PM2.5 Analysis

This notebook extracts image-based visual features from a fixed Region of Interest (ROI B) selected from webcam images.

Compared with the original sky/ground feature extraction method, this notebook adopts **Method 2: Whole-ROI Feature Extraction**.  
Instead of splitting the image into sky and ground regions, features are extracted from the entire selected ROI.

The extracted features include:

- Mean RGB values: `R_roi`, `G_roi`, `B_roi`
- Mean saturation: `S_mean`
- Blue-to-Red ratio: `B_R_ratio`
- Image contrast: `contrast`

The output is saved as:

`Processing Outputs/image_feature_roi.csv`

In [1]:
import re
from pathlib import Path
from PIL import Image
import numpy as np
import csv
from datetime import datetime
from matplotlib.colors import rgb_to_hsv

## Helper Functions

These functions are used to:

1. Load each image as an RGB NumPy array.
2. Crop the selected ROI B.
3. Compute visual features from the whole ROI.

In [2]:
def load_rgb_array(p: Path) -> np.ndarray:
    """
    Load image and return an RGB array with shape (H, W, 3).
    """
    img = Image.open(p).convert("RGB")
    return np.array(img)


def crop_roi(arr: np.ndarray, top: int, left: int, height: int, width: int) -> np.ndarray:
    """
    Crop a fixed rectangular ROI from the image.

    Parameters:
    - top: starting row index
    - left: starting column index
    - height: ROI height
    - width: ROI width
    """
    return arr[top:top + height, left:left + width, :]


def mean_rgb(arr: np.ndarray):
    """
    Compute mean R, G, B values for the ROI.
    """
    if arr.size == 0:
        return (np.nan, np.nan, np.nan)

    r = arr[:, :, 0].mean()
    g = arr[:, :, 1].mean()
    b = arr[:, :, 2].mean()

    return (float(r), float(g), float(b))


def mean_saturation(arr: np.ndarray):
    """
    Compute mean saturation from the HSV color space.

    RGB values are first normalized to [0, 1],
    then converted to HSV.
    """
    if arr.size == 0:
        return np.nan

    arr_norm = arr / 255.0
    hsv = rgb_to_hsv(arr_norm)
    saturation = hsv[:, :, 1]

    return float(saturation.mean())


def blue_red_ratio(arr: np.ndarray):
    """
    Compute the Blue-to-Red ratio: mean(B) / mean(R).

    A small epsilon is used to avoid division by zero.
    """
    if arr.size == 0:
        return np.nan

    r_mean = arr[:, :, 0].mean()
    b_mean = arr[:, :, 2].mean()

    eps = 1e-6
    return float(b_mean / (r_mean + eps))


def image_contrast(arr: np.ndarray):
    """
    Compute image contrast using the standard deviation of grayscale intensity.

    Higher values indicate stronger intensity variation within the ROI.
    """
    if arr.size == 0:
        return np.nan

    gray = arr.mean(axis=2)
    return float(gray.std())

## Parameters and Paths

The input folder is scanned recursively.  
Only image files with filenames in the format `YYYYMMDD-HHMM.jpg` are processed.

ROI B is used as the fixed region for feature extraction:

- top = 160
- left = 116
- height = 380
- width = 515

In [3]:
root = Path("Images")
output_csv = Path("Processing Outputs/image_feature_roi.csv")

# ROI B parameters
ROI_TOP = 160
ROI_LEFT = 116
ROI_HEIGHT = 380
ROI_WIDTH = 515

IMG_EXTS = {".jpg", ".jpeg", ".JPG", ".JPEG"}

# Expected filename format: YYYYMMDD-HHMM
name_re = re.compile(r"^(?P<date>\d{8})-(?P<hour>\d{4})$")

## Feature Extraction Loop

For each valid image:

1. Check file extension and filename format.
2. Parse date and time from the filename.
3. Load the image as RGB.
4. Check whether the image is large enough for ROI cropping.
5. Crop ROI B.
6. Extract whole-ROI features.
7. Append one row to the output table.

In [4]:
rows = []
skipped_files = []

for p in sorted(root.rglob("*")):
    # Check image extension
    if not (p.is_file() and p.suffix in IMG_EXTS):
        continue

    # Check filename pattern
    stem = p.stem
    if not name_re.match(stem):
        skipped_files.append((str(p), "Invalid filename format"))
        continue

    # Parse datetime from filename
    try:
        dt = datetime.strptime(stem, "%Y%m%d-%H%M")
    except ValueError:
        skipped_files.append((str(p), "Datetime parsing failed"))
        continue

    date_csv = dt.strftime("%Y-%m-%d")
    time_csv = dt.strftime("%H:%M")

    # Load image
    try:
        arr = load_rgb_array(p)
    except Exception as e:
        skipped_files.append((str(p), f"Image loading failed: {e}"))
        continue

    H, W, C = arr.shape

    # Check whether ROI B fits inside the image
    if (
        C != 3
        or H < ROI_TOP + ROI_HEIGHT
        or W < ROI_LEFT + ROI_WIDTH
    ):
        skipped_files.append((str(p), "Image too small for ROI cropping"))
        continue

    # Crop ROI B
    roi = crop_roi(arr, ROI_TOP, ROI_LEFT, ROI_HEIGHT, ROI_WIDTH)

    # Extract features
    r_roi, g_roi, b_roi = mean_rgb(roi)
    s_mean = mean_saturation(roi)
    br_ratio = blue_red_ratio(roi)
    contrast = image_contrast(roi)

    # Store row
    rows.append({
        "date": date_csv,
        "hour": time_csv,
        "R_roi": r_roi,
        "G_roi": g_roi,
        "B_roi": b_roi,
        "S_mean": s_mean,
        "B_R_ratio": br_ratio,
        "contrast": contrast,
        "image_path": str(p)
    })

## Export Features to CSV

The extracted features are saved to:

`Processing Outputs/image_feature_roi.csv`

Each row corresponds to one webcam image.

In [5]:
output_csv.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "date",
    "hour",
    "R_roi",
    "G_roi",
    "B_roi",
    "S_mean",
    "B_R_ratio",
    "contrast",
    "image_path"
]

with output_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Done. Wrote {len(rows)} rows to {output_csv}")
print(f"Skipped {len(skipped_files)} files.")

Done. Wrote 743 rows to Processing Outputs/image_feature_roi.csv
Skipped 0 files.
